In [1]:
import ee 
from RadGEEToolbox import GenericCollection, LandsatCollection, get_palette
# import GEE_UBM
from GEE_UBM import InputCollections, SnowMeltCollection, build_model_ready_collection, OriginalUBMRun, ModifiedUBM1Run
import geemap
import numpy as np

In [2]:
service_account = 'localpythonscripts@ut-gee-ugs-bsf-dev.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, 'C:\\Users\\mradwin\\ut-gee-ugs-bsf-dev-53dcc5d729e0.json')
ee.Initialize(credentials=credentials)

In [3]:
base = SnowMeltCollection(start_date='2024-01-01', end_date='2024-12-30', geometry=ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry())

In [4]:
delta_swe = base.calculate_daily_delta_swe()
dates = delta_swe.dates
print(dates)
print(delta_swe.image_grab(0).bandNames().getInfo())

['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08', '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14', '2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20', '2024-01-21', '2024-01-22', '2024-01-23', '2024-01-24', '2024-01-25', '2024-01-26', '2024-01-27', '2024-01-28', '2024-01-29', '2024-01-30', '2024-01-31', '2024-02-01', '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05', '2024-02-06', '2024-02-07', '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13', '2024-02-14', '2024-02-15', '2024-02-16', '2024-02-17', '2024-02-18', '2024-02-19', '2024-02-20', '2024-02-21', '2024-02-22', '2024-02-23', '2024-02-24', '2024-02-25', '2024-02-26', '2024-02-27', '2024-02-28', '2024-02-29', '2024-03-01', '2024-03-02', '2024-03-03', '2024-03-04', '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08', '2024-03-09', '2024-03-10', '2024-03-11', '2024

In [15]:
base_class = InputCollections(start_date=f'2024-01-01', end_date=f'2024-12-31', soil_thickness_raster='gNATSGO_filled')
precip = base_class.get_precip('DAYMET_daily_precip')
precip_dates = precip.dates
print(precip_dates)
DAYMET_daily_precip = GenericCollection(collection=ee.ImageCollection("NASA/ORNL/DAYMET_V4").select(['prcp']), start_date=f'2024-01-01', end_date=f'2024-12-31')\
                                    .mask_to_polygon(ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry()).band_rename('prcp', 'precipitation')
PRISM_daily_precip = GenericCollection(collection=ee.ImageCollection("OREGONSTATE/PRISM/ANd").select(['ppt']), start_date=f'2024-01-01', end_date=f'2024-12-31')\
                                    .mask_to_polygon(ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry()).band_rename('ppt', 'precipitation')

['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08', '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14', '2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20', '2024-01-21', '2024-01-22', '2024-01-23', '2024-01-24', '2024-01-25', '2024-01-26', '2024-01-27', '2024-01-28', '2024-01-29', '2024-01-30', '2024-01-31', '2024-02-01', '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05', '2024-02-06', '2024-02-07', '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13', '2024-02-14', '2024-02-15', '2024-02-16', '2024-02-17', '2024-02-18', '2024-02-19', '2024-02-20', '2024-02-21', '2024-02-22', '2024-02-23', '2024-02-24', '2024-02-25', '2024-02-26', '2024-02-27', '2024-02-28', '2024-02-29', '2024-03-01', '2024-03-02', '2024-03-03', '2024-03-04', '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08', '2024-03-09', '2024-03-10', '2024-03-11', '2024

In [49]:
print(ee.Number(PRISM_daily_precip.image_grab(0).projection().nominalScale()).getInfo())

4638.312116423505


In [25]:
soil_water_input = base.calculate_daily_soil_input(precip_collection=precip, delta_swe_collection=delta_swe)
soil_water_input_dates = soil_water_input.dates
print(soil_water_input_dates)
print(soil_water_input.image_grab(0).bandNames().getInfo())

monthly_soil_water = soil_water_input.monthly_sum_collection

['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08', '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14', '2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19', '2024-01-20', '2024-01-21', '2024-01-22', '2024-01-23', '2024-01-24', '2024-01-25', '2024-01-26', '2024-01-27', '2024-01-28', '2024-01-29', '2024-01-30', '2024-01-31', '2024-02-01', '2024-02-02', '2024-02-03', '2024-02-04', '2024-02-05', '2024-02-06', '2024-02-07', '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13', '2024-02-14', '2024-02-15', '2024-02-16', '2024-02-17', '2024-02-18', '2024-02-19', '2024-02-20', '2024-02-21', '2024-02-22', '2024-02-23', '2024-02-24', '2024-02-25', '2024-02-26', '2024-02-27', '2024-02-28', '2024-02-29', '2024-03-01', '2024-03-02', '2024-03-03', '2024-03-04', '2024-03-05', '2024-03-06', '2024-03-07', '2024-03-08', '2024-03-09', '2024-03-10', '2024-03-11', '2024

In [26]:
print(monthly_soil_water.dates)

['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']


In [28]:
index = -3
Map = geemap.Map(center=[39.5, -111.5], zoom=7)
Map.addLayer(delta_swe.image_grab(index), vis_params={'bands':'delta_swe', 'min': -10, 'max': 10, 'palette': get_palette('rdbu')}, name='Delta SWE')
Map.addLayer(precip.image_grab(index), vis_params={'bands':'precipitation', 'min': 0, 'max': 20, 'palette': get_palette('blues')}, name='Precipitation')
# Map.addLayer(PRISM_daily_precip.image_grab(index), vis_params={'bands':'precipitation', 'min': 0, 'max': 20, 'palette': get_palette('blues')}, name='PrecipitationPrism')
Map.addLayer(soil_water_input.image_grab(index), vis_params={'bands':'precip_and_snowmelt_input', 'min': 0, 'max': 20, 'palette': get_palette('blues')}, name='Soil Water Input')
# Map.addLayer(ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry())
Map.addLayer(monthly_soil_water.image_grab(-1), vis_params={'bands':'precip_and_snowmelt_input', 'min': 0, 'max': 600, 'palette': get_palette('blues')}, name='Monthly Soil Water Input')
print(dates[index])
Map

2024-12-28


Map(center=[39.5, -111.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [3]:
run_loop = True
##### LOOP FOR HANDLING SNODAS + DAYMET PRECIP COMBINATION EXPORTS #####
# if run_loop:
#     years = np.arange(2015, 2025, 1)
#     for year in years:
#         base = SnowMeltCollection(start_date=f'{year}-01-01', end_date=f'{year}-12-31', geometry=ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry())
#         delta_swe = base.calculate_daily_delta_swe()
#         swe_dates = delta_swe.dates
#         base_class = InputCollections(start_date=f'{year}-01-01', end_date=f'{year}-12-31', soil_thickness_raster='gNATSGO_filled')
#         precip = base_class.get_precip('DAYMET_daily_precip')
#         precip_dates = precip.dates
#         soil_water_input = base.calculate_daily_soil_input(precip_collection=precip, delta_swe_collection=delta_swe)
#         soil_water_input_dates = soil_water_input.dates
#         monthly_soil_water_input = soil_water_input.monthly_sum_collection
#         monthly_soil_water_input_dates = monthly_soil_water_input.dates

#         export = monthly_soil_water_input.export_to_asset_collection(asset_collection_path='projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_DAYMET_PRECIP_PLUS_SNOWMELT_1KM',
#                                                                     region=ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry(),
#                                                                     scale=1000,
#                                                                     dates=monthly_soil_water_input_dates)


##### LOOP FOR HANDLING SNODAS + PRISM PRECIP COMBINATION EXPORTS #####
if run_loop:
    years = np.arange(2015, 2025, 1)
    for year in years:
        base = SnowMeltCollection(start_date=f'{year}-01-01', end_date=f'{year}-12-31', geometry=ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry())
        delta_swe = base.calculate_daily_delta_swe()
        swe_dates = delta_swe.dates
        base_class = InputCollections(start_date=f'{year}-01-01', end_date=f'{year}-12-31', soil_thickness_raster='gNATSGO_filled')
        precip = base_class.get_precip('PRISM_daily_precip')
        precip_dates = precip.dates
        precip_scale = ee.Number(precip.image_grab(0).projection().nominalScale())
        soil_water_input = base.calculate_daily_soil_input(precip_collection=precip, delta_swe_collection=delta_swe)
        soil_water_input_dates = soil_water_input.dates
        monthly_soil_water_input = soil_water_input.monthly_sum_collection
        monthly_soil_water_input_dates = monthly_soil_water_input.dates

        export = monthly_soil_water_input.export_to_asset_collection(asset_collection_path='projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM',
                                                                    region=ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry(),
                                                                    scale=precip_scale.getInfo(),
                                                                    dates=monthly_soil_water_input_dates)

Queued 12 export tasks to projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM
Queued 12 export tasks to projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM
Queued 12 export tasks to projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM
Queued 12 export tasks to projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM
Queued 12 export tasks to projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM
Queued 12 export tasks to projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM
Queued 12 export tasks to projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM

In [4]:
snodas_daymet = GenericCollection(collection=ee.ImageCollection('projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_DAYMET_PRECIP_PLUS_SNOWMELT_1KM'), 
                                  start_date='2024-01-01', end_date='2024-12-31')
snodas_daymet_dates = snodas_daymet.dates
print(snodas_daymet_dates)
print(snodas_daymet.image_grab(0).bandNames().getInfo())

snodas_prism = GenericCollection(collection=ee.ImageCollection('projects/ut-gee-ugs-bsf-dev/assets/UT_Precip_and_Snowmelt_Image_Collections/UT_SNODAS_PRISM_PRECIP_PLUS_SNOWMELT_5KM'), 
                                  start_date='2024-01-01', end_date='2024-12-31')
snodas_prism_dates = snodas_prism.dates
print(snodas_prism_dates)
print(snodas_prism.image_grab(0).bandNames().getInfo())

['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']
['precip_and_snowmelt_input']
['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']
['precip_and_snowmelt_input']


In [6]:
index = -1
Map = geemap.Map(center=[39.5, -111.5], zoom=7)
Map.addLayer(snodas_daymet.image_grab(index), vis_params={'bands':'precip_and_snowmelt_input', 'min': 0, 'max': 300, 'palette': get_palette('blues')}, name='DAYMET + SNODAS')
Map.addLayer(snodas_prism.image_grab(index), vis_params={'bands':'precip_and_snowmelt_input', 'min': 0, 'max': 300, 'palette': get_palette('blues')}, name='PRISM + SNODAS')
print(snodas_daymet_dates[index])
Map

2024-12-01


Map(center=[39.5, -111.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…